In [ ]:
import sys
import os
from pathlib import Path

import torch
import torch.nn as nn

sys.path.append(os.path.abspath("../src"))

from models import SimpleCNN, ResNet18, DenseNet121, EfficientNetB0, MobileNetV2, ShuffleNetV2, SqueezeNet

from data import (
    prepare_full_dataframe, 
    prepare_data, 
    sample_image_path,
    get_transforms
)

from train_eval import (
    setup_training,
    train_model,
    evaluate,
    predict_single_image
)

from utils import (
    get_device,
    plot_training_history,
    plot_training_history_compact,
    plot_confusion_matrix_figure,
    get_model_path
)

from train import run_training_pipeline

from experiement_types import Metrics, History, ModelOutput

import config

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# Data Preprocessing

In [ ]:
dataset_path = config.DATASET_PATH
print("Dataset location:", dataset_path)

metadata_file = os.path.join(dataset_path, "Data_Entry_2017.csv")
df = prepare_full_dataframe(metadata_file, dataset_path)

print("Total images:", len(df))
print("Unique patients:", df["Patient ID"].nunique())
df["Finding Labels"].value_counts().head()

In [ ]:
print(df["split"].value_counts())

In [ ]:
train_loader, val_loader, test_loader = prepare_data(df)
device = get_device()

# Model Individual Training/Testing

In [ ]:
model_registry = {
    "SimpleCNN": SimpleCNN,
    "ResNet18": lambda: ResNet18(num_classes=2, in_channels=1),
    "DenseNet121": lambda: DenseNet121(num_classes=2, in_channels=1),
    "EfficientNet-B0": lambda: EfficientNetB0(num_classes=2, in_channels=1),
    "MobileNetV2": lambda: MobileNetV2(num_classes=2, in_channels=1),
    "ShuffleNetV2": lambda: ShuffleNetV2(num_classes=2, in_channels=1),
    "SqueezeNet": lambda: SqueezeNet(num_classes=2, in_channels=1)
}
model_names = list(model_registry.keys())
model_builders = list(model_registry.values())

experiment_outputs = {}

## Simple CNN

In [ ]:
metrics, history = run_training_pipeline(model_names[0], model_builders[0], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[0]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

## ResNet-18

In [ ]:
metrics, history = run_training_pipeline(model_names[1], model_builders[1], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[1]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

## DenseNet-121

In [ ]:
metrics, history = run_training_pipeline(model_names[2], model_builders[2], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[2]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

## EfficientNet-B0

In [ ]:
metrics, history = run_training_pipeline(model_names[3], model_builders[3], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[3]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

## MobileNet-V2

In [ ]:
metrics, history = run_training_pipeline(model_names[4], model_builders[4], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[4]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

## ShuffleNetV2

In [ ]:
metrics, history = run_training_pipeline(model_names[5], model_builders[5], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[5]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

## SqueezeNet

In [ ]:
metrics, history = run_training_pipeline(model_names[6], model_builders[6], train_loader, val_loader, test_loader, device, live_plot=True)
experiment_outputs[model_names[6]] = ModelOutput(
    metrics=Metrics(**metrics),
    history=History(**history)
)

# Model Comparison Plots
Visualize the test results for all models using bar plots for accuracy, precision, recall, and F1 score.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame([
    vars(out.metrics) for out in experiment_outputs.values()
])

metrics = ["accuracy", "precision", "recall", "f1", "auprc"]

plt.figure(figsize=(12, 6))
results_melted = results_df.melt(
    id_vars="model",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
)
sns.barplot(data=results_melted, x="model", y="Score", hue="Metric")
plt.title("Model Comparison on Test Set")
plt.ylabel("Score")

min_score = results_melted["Score"].min()
max_score = results_melted["Score"].max()
padding = max(0.01, 0.08 * (max_score - min_score))
ymin = max(0.0, min_score - padding)
ymax = min(1.0, max_score + padding)
plt.ylim(ymin, ymax)

plt.legend(loc="lower right")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()